# **[ <small>HOMEWORK DATA SCIENCE</small> ] | Crawl data**
<br><br>

## **SET UP ENVIRONMENT**

---


In [ ]:
!pip install beautifulsoup4 cloudscraper numpy pandas playwright lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.7/99.7 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 MB 16.9 MB/s eta 0:00:00


In [ ]:
!playwright install chromium

173.7 MiB [] 0% 0.0s173.7 MiB [] 0% 28.1s173.7 MiB [] 0% 17.7s173.7 MiB [] 0% 8.5s173.7 MiB [] 1% 5.3s173.7 MiB [] 1% 4.1s173.7 MiB [] 2% 3.3s173.7 MiB [] 3% 2.9s173.7 MiB [] 4% 2.6s173.7 MiB [] 5% 2.6s173.7 MiB [] 5% 2.8s173.7 MiB [] 6% 2.8s173.7 MiB [] 7% 2.7s173.7 MiB [] 8% 2.6s173.7 MiB [] 8% 2.5s173.7 MiB [] 9% 2.5s173.7 MiB [] 10% 2.3s173.7 MiB [] 11% 2.3s173.7 MiB [] 12% 2.2s173.7 MiB [] 13% 2.2s173.7 MiB [] 14% 2.1s173.7 MiB [] 15% 2.0s173.7 MiB [] 16% 1.9s173.7 MiB [] 18% 1.8s173.7 MiB [] 19% 1.8s173.7 MiB [] 19% 1.7s173.7 MiB [] 20% 1.7s173.7 MiB [] 22% 1.6s173.7 MiB [] 23% 1.6s173.7 MiB [] 24% 1.5s173.7 MiB [] 25% 1.5s173.7 MiB [] 25% 1.6s173.7 MiB [] 26% 1.6s173.7 MiB [] 27% 1.6s173.7 MiB [] 28% 1.5s173.7 MiB [] 29% 1.5s173.7 MiB [] 30% 1.4s173.7 MiB [] 32% 1.4s173.7 MiB [] 33% 1.3s173.7 MiB [] 35% 1.2s173.7 MiB [] 36% 1.2s173.7 MiB [] 38% 1.1s173.7 MiB [] 39% 1.1s173.7 MiB [] 41% 1.1s173.7 MiB [] 42% 1.0s173.7 MiB [] 44% 1.0s173.7 MiB [] 45% 0.9s173.7 MiB [] 47% 0.9s173.7 

In [ ]:
'''
IMPORT LIBRARIES
'''
from bs4 import BeautifulSoup
import cloudscraper, random, time, json, os, requests
import argparse
from playwright.async_api import async_playwright

## **GET HTML FROM A URL**

---

In [ ]:
'''
GET HTML
'''
def get_html_pass_cloudflare(
        url: str,
        state_file: str = 'cookies.json',
        domain_name = 'https://batdongsan.com.vn/') -> str:
    '''
    Function to get html document text --> Return: html text
    '''

    if not url.startswith('http'):
        return None

    # list of user-agent that mimics the behaviors of real users
    user_agents = [
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
        'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.2 Safari/605.1.15',
        'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
    ]

    # create headers which simulate the real header, randomly using the user-agents created
    headers = {
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
        'Accept-Language': 'vi-VN,vi;q=0.9,en-US;q=0.8,en;q=0.7,fr;q=0.6',
        'Accept-Encoding': 'gzip, deflate, br',
        'DNT': '1',
        'Connection': 'keep-alive',
        'Sec-Fetch-Mode': 'navigate',
        'Sec-Fetch-Site': 'none',
        'Sec-Fetch-User': '?1',
        'Cache-Control': 'max-age=0',
        'sec-ch-ua': '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
        'sec-ch-ua': '"Not_A Brand";v="8", "Chromium";v="120", "Google Chrome";v="120"',
        'sec-ch-ua-platform': '"macOS"',
        'User-Agent': random.choice(user_agents),
        'Referer': domain_name,
        'Origin': domain_name
    }
    scraper = cloudscraper.create_scraper()

    # Load cookies
    if state_file and os.path.exists(state_file):
        with open(state_file, "r") as f:
            cookies = json.load(f)
            scraper.cookies.update(cookies)


    # Try up to 3 times if response is not accepted
    for attempt in range(3):
        try:
            time.sleep(random.uniform(1,3))

            response = scraper.get(url, headers = headers, timeout = random.randint(100,300))

            # save cookies
            if state_file:
                with open(state_file, "w") as f:
                    json.dump(requests.utils.dict_from_cookiejar(scraper.cookies), f)

            # Check response status
            if response.status_code in (429, 503) or "just a moment" in response.text.lower():
                # Try to use other user-agent
                headers['User-Agent'] = random.choice(user_agents)
                time.sleep(random.uniform(2,5))
                continue

            elif response.status_code == 200:
                html_text = response.text
                return html_text

            elif attempt == 2:
                print("[|X| FAIL] Cloudflare block this request")
                return None

        except Exception as e:
            print("[!!!| EXCEPTION] Exception : {}".format(e))
            return None

In [ ]:
'''
PARSE HTML
'''
def parse_html(html_text):
    '''
    This function works for parsing http document
    '''
    from bs4 import BeautifulSoup
    return BeautifulSoup(html_text, 'lxml')

## **GET INFORMATION FROM A HTML DOCUMENT**

---

In [ ]:
'''
GET TITLE
'''
def get_title(html_parse: BeautifulSoup) -> list:
    '''
    Function gets the title of the articles after parsing to BeautifulSoup object
    --> Return : List of title
    '''
    return [html_parse.title.text]

In [ ]:
'''
GET PUBLISHER
'''
def get_publisher(html_parse: BeautifulSoup) -> list:
    '''
    Function gets the publisher, or list of publisher of the document
    Publisher is a person or an organization publishing the article, not the author
    --> Return : List of publisher
    '''
    selector = '#__next main > div.ArticlePageTemplate_articlePageContainer__wcRoZ.container > div:nth-child(2) > div > div > div > div.AuthorInfo_authorName__m9KD3 > a'
    publisher_ls = html_parse.select(selector)
    if len(publisher_ls):
        return [publisher.text for publisher in publisher_ls]
    return None


In [ ]:
'''
GET LATEST UPDATE TIME
'''
def get_latest_update(html_parse: BeautifulSoup) -> list:
    '''
    Function gets the latest update time
    --> Return: list of update time
    '''
    selector = '#__next main > div.ArticlePageTemplate_articlePageContainer__wcRoZ.container > div:nth-child(2) > div > div > div > div.AuthorInfo_postDate__UTKIr'
    update_time_ls = html_parse.select(selector)

    if len(update_time_ls):
        return [update_time.text for update_time in update_time_ls]
    return None

In [ ]:
'''
GET CONTENT, AUTHOR, ORIGINAL LINKS
'''
def get_content(html_parse: BeautifulSoup) -> dict:
    '''
    Function gets the article's content
    --> Return: dict: {
        "content"       : str,
        "author"        : str,
        "source"        : str,
        "publish time"  : str,
        "link source"   : str
    }
    '''
    selector = '#__next main > div.ArticlePageTemplate_articlePageContainer__wcRoZ.container > div:nth-child(4) > div.col-xl-8.col-lg-8.col-md-12.col-12 > article > div:nth-child(1) > *'
    content_list = html_parse.select(selector)
    if len(content_list):
        full_content = ''
        accept_tag = ['div', 'h1', 'h2', 'h3', 'h4', 'h5'] # list of acceptable tags, this is for eleminating the figure
        end_main_content = ['—','——','———',]               # list of signals which indicate that the main content ends

        author = 'Unknown'
        source = 'Unknown'
        publish_time = 'Unknown'
        source_link = 'Unknown'

        # Extract full content, full content ends by the line '——'
        for i in range(len(content_list)):
            if content_list[i].name in accept_tag and '—' not in content_list[i].text :
                full_content += content_list[i].text + "\n\n"

            # Extract author and original article:
            # after main content, the information of author and the original article is shown in (i+1)-th element
            # ( Because the i-th element is the line '——')
            elif '—' in content_list[i].text and i+1 < len(content_list):
                # format of information is:
                # <p>Tác giả:...<br>Nguồn:...<br>
                # split it into separate parts: ['Tác giả:...', <br>, 'Nguồn:...', <br>,...] by method `children`
                for child in content_list[i+1].children:
                    if child.name != "br":
                        separate_index = child.text.find(":")+1
                        info = child.text[separate_index:]
                        format_text = " ".join(info.split())
                        if 'tác giả' in child.text.lower():
                            author = format_text
                        if 'nguồn tin' in child.text.lower():
                            source = format_text
                        if 'thời gian' in child.text.lower():
                            publish_time = format_text
                        if 'link' in child.text.lower():
                            source_link = format_text
                break

        return {
            "content": full_content,
            "author" : author,
            "source" : source,
            "publish time": publish_time,
            "source link": source_link
        }

    return None


In [ ]:
'''
GET FULL INFORMATION FROM HTML
'''
def get_full_information(html_parse) -> dict:
    '''
    Function gets full information of an article
    --> Return: dict:
        {
            "title"                 : str,
            "publisher"             : str,
            "lastest update time"   : str,
            "content"               : str,
            "author"                : str,
            "source"                : str,
            "publish time"          : str,
            "link source"           : str
        }
    '''
    # Get title
    print("\r       Getting title...", end = "")
    title = get_title(html_parse)
    if title:
        title = title[0]
    else:
        title = 'Unknown'
        print("\r     |X| Cannot get the title !!!")

    # Get publisher
    print("\r       Getting publisher...", end = "")
    publiser = get_publisher(html_parse)
    if publiser:
        publiser = ", ".join(publiser)
    else:
        print(("\r    |X| Cannot get the publisher !!!"))
        publiser = 'Unknown'

    # Get latest update time
    print("\r       Getting latest update time...", end="")
    latest_update_time = get_latest_update(html_parse)
    if latest_update_time:
        latest_update_time = " ".join(latest_update_time)
    else:
        print(("\r    |X| Cannot get the latest update time !!!"))
        latest_update_time = 'Unknown'

    # Get content, author, source, publish time, source link
    result = get_content(html_parse)
    print("\r       Getting content, author, source, publish time...", end = "")
    if result:
        result["title"] = title
        result["publisher"] = publiser
        result["latest update time"] = latest_update_time
        print("\r       Finish crawling!                                     ")

    else:
        print("\r    |X| Cannot get the content, author, source, publish time!!!")
        return {
            "title"                 : title,
            "publisher"             : publiser,
            "lastest update time"   : latest_update_time,
            "content"               : 'Unknown',
            "author"                : 'Unknown',
            "source"                : 'Unknown',
            "publish time"          : 'Unknown',
            "link source"           : 'Unknown'
        }

    return result

## **GET URLS LIST**

---

Access the tab `tintuc` of website to get links list. But the number of links shown is limited, to get more links, we have to interact with the website by clicking the button `Xem thêm ` at the bottom of the page. For automatically interacting with the page, I use `playwright` to simulate the user behaviours of browsing the web.

In [ ]:
import os
from pathlib import Path
from urllib.parse import urlunparse

'''
CLASS DEFINING THE BEHAVIOURS OF BROWSING WEB
'''
class BrowserController():
    def __init__(self, p):
        self.page = None
        self.browser = None
        self.p = p

    async def open_browser(self):
        '''
        This function open browser and start a new sesion
        '''
        try:
            self.browser = await self.p.chromium.launch(headless = True)
            self.page = await self.browser.new_page()
            return self.browser, self.page

        except Exception as e:
            print("|X| Exception in open new session: {}".format(e))
            return None

    async def access_html(self, html: str):
        '''
        This function accesses and shows html file to extract information
        '''
        # Check if parameter html is a file
        if html.endswith(".html"):
            # Get the absolute path to current file
            base_dir = Path(html)
            file_path = base_dir / html

            if os.path.exists(file_path):
                print("/ Showing html file {} ...".format(html))
                # format file path to the form which can be read be playwright
                file_url = urlunparse(('file', '', str(file_path).replace(os.path.sep, '/'), '', '', ''))
                await self.page.goto(file_url, wait_until="networkidle")
                return True
            else:
                print("|X| (!_!) Access Fail: File {} is not existed".format(file_path))
                return False

        elif html.startswith("http"):
                try:
                    print("/ Accessing the url {} ...".format(html))
                    await self.page.goto(html, wait_util="networkidle")
                    return True
                except Exception as e:
                    print("|X| (*_*) Access Fail: {}".format(e))
                    return False

In [ ]:
'''
INTERACT WITH THE PAGE BY CLICKING THE BUTTON
'''
async def interact_html_getting_links(page, num_time_click=3):
        '''
        This function will interact the html to show more information.
        This will help with adding more elements into document
        '''
        button_selector = '#__next > main > div > div:nth-child(3) > div.col-xl-8.col-lg-8.col-md-12.col-12 > div:nth-child(4) button.ArticleFeed_showMoreButton__beGxM'

        for _ in range(num_time_click):
                button_locator = page.locator(button_selector)
                await button_locator.scroll_into_view_if_needed()
                await button_locator.click()
                await page.wait_for_load_state("load")
                await page.wait_for_timeout(1000)

        return await page.content()

In [ ]:
'''
GETTING LINK LIST
'''
async def get_link_list(url: str, save_file = "/kaggle/working/tem.html"):
        '''
        This function gets list of links of a website
        '''
        print("------ GETTING LISTS OF URLS -----")
        # If there is no html file which containing list of urls, connect to website to get html document
        if not os.path.exists(save_file):
                html_text = get_html_pass_cloudflare(url)
                if html_text:
                        with open(save_file, "w", encoding = "utf-8") as f:
                                f.write(html_text)
                else:
                        print("(*_*) Cannot access the website !!")
                        return None

        # open browser
        async with async_playwright() as p:
                browserController = BrowserController(p)
                # open browser
                result_open = await browserController.open_browser()
                if result_open:
                        browser, page = result_open
                        if await browserController.access_html(save_file):
                                html_text = await interact_html_getting_links(page)
                        else: html_text = None
                else: html_text = None

        if html_text:
                html_parse = parse_html(html_text)
                main_area = '#__next > main > div > div:nth-child(2) > div.col-lg-8.col-md-12.col-sm-12 > a'
                side_bar = '#__next > main > div > div:nth-child(2) > div.col-lg-4.col-md-12.col-sm-12'
                best_views = '#__next > main > div > div:nth-child(3) > div.hidden-xs.col-xl-4.col-lg-4.col-md-12.col-12 > div.PopularArticles_popularArticlesWrapper__VP0DZ'
                ls_container = '#__next > main > div > div:nth-child(3) > div.col-xl-8.col-lg-8.col-md-12.col-12 > div:nth-child(4) > div.ArticleCardLarge_articleWrapper__rp8cl > div > div:nth-child(1) > a'

                ls_links = []

                # get link in main area
                main_area_link = html_parse.select(main_area)
                if len(main_area_link):
                        ls_links.append(html_parse.select(main_area)[0].get('href'))
                else: print("|X| Cannot get the link in the main area !!!")

                # get links in side bar
                elements_side_bar_ls = html_parse.select(side_bar)
                if elements_side_bar_ls:
                        element_a_ls = elements_side_bar_ls[0].find_all('a', href = True)
                        if len(element_a_ls):
                                ls_links.extend([element_a.get('href') for element_a in element_a_ls])
                        else: print("(!!!) | There is no articles in side bar area !!!")
                else: print("|X| Cannot get the links in the side bar")

                # get links in best views
                best_view_ls = html_parse.select(best_views)
                if len(best_view_ls):
                        best_view_articles = best_view_ls[0].find_all('a', href = True)
                        if len(best_view_articles):
                                ls_links.extend([article.get('href') for article in best_view_articles])
                        else: print("(!!!) | There is no articles in the best view area!!!")
                else: print("|X| Cannot get the links in the best view area!!!")

                # get links in ls_container
                articles = html_parse.select(ls_container)
                if len(articles):
                        for element_a in articles:
                                if element_a:
                                        ls_links.append(element_a.get('href'))
                else: print("|X| Cannot get the links in container!!!")

                if len(ls_links): return ls_links
                return None

        print("(*_*) Cannot access the website !!")
        return None

## **CRAWLING DATA**

---

In [ ]:
'''
CRAWLING DATA FUNCTION
'''
async def crawl(links_list_url: str, destination = "batdongsan.json", url_html = "tem.html"):
        '''
        Function crawls data and save data into the destination
        '''
        # check if the destination file has some data?
        if os.path.exists(destination):
                with open(destination, "r", encoding = "utf-8") as f:
                        prev_data = json.load(f)
        else: prev_data = []
        # get list of links
        links_ls = await get_link_list(links_list_url, url_html)
        if links_ls:
                print("Successfully getting {} urls".format(len(links_ls)))
                # filter the unique links
                url_set = set(links_ls)
                for url in links_ls:
                        if url in url_set:
                                url = "https://batdongsan.com.vn" + url
                                html_text = get_html_pass_cloudflare(url)
                                if html_text:
                                        html_parse = parse_html(html_text)
                                        # get information in html
                                        info = get_full_information(html_parse)
                                        if info:
                                                info["url"] = url
                                                prev_data.append(info)
                                        else: print("|X| Cannot get information in url {}".format(url))
                        else: print("|X| Cannot access the article {}".format(url))
                save_data(prev_data, destination)

        else: print("|X| Cannot get link list !!!")


## **SAVE DATA**

---

In [ ]:
'''
SAVE DATA
'''
def save_data(ls_of_dict_object, save_file = "batdongsan.json"):
        '''
        This function saves the crawled data into the destination
        '''
        hierachy = save_file.split("/")
        if save_file.startswith("/"):
            hierachy[0] = "/"
        save_folder = os.path.join(*hierachy[:-1])
        print(save_folder)
        os.makedirs(save_folder, exist_ok = True)
        with open(save_file, "w", encoding = "utf-8") as f:
            json.dump(ls_of_dict_object, f, ensure_ascii = False, indent = 4)

## **MAIN CRAWLING DATA**

---

In [ ]:
link_list_url = 'https://batdongsan.com.vn/tin-tuc'
save_path = '/content/data/data.json'
link_list_html_file = '/content/tem.html'

await crawl(link_list_url, save_path , link_list_html_file)

------ GETTING LISTS OF URLS -----
/ Showing html file /content/MyDrive/MyDrive/DataScience-Lab/Lab01/tem.html ...
Successfully getting 30 urls
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
       Finish crawling!                                     
  

Extract an excel file

In [ ]:
import os
import json
import pandas as pd

%cd /content/data
data_path = "data.json"

with open(data_path, "r", encoding = "utf-8") as f:
  articles_ls = json.load(f)

title_ls = []
url_ls = []

for article in articles_ls:
  title_ls.append(article["title"])
  url_ls.append(article["url"])

df_data = pd.DataFrame(
    {
        "title": title_ls,
        "url": url_ls
    }
)

excel_file = "data.xlsx"
df_data.to_excel(excel_file, index = False)

/content/data
